# Historical archive: GPT2

**Status: not independently reproduced.**

Checkpoint / purpose: `microsoft/DialogRPT-updown (loaded by GPT2.ipynb)`.

Historical code, not independently reproduced. Saved settings can differ from report labels; notebook filenames are not authoritative configurations. The source named GPT2.ipynb loads DialogRPT, so it does not substantiate a GPT-2 Medium run. Principal training code evaluates on validation; a final held-out test run is not established.

Publication edits only: credentials use `HF_TOKEN`; personal absolute paths were relocated; saved outputs, attachments and runtime metadata were removed. Historical installation cells and local data dependencies are retained. Review code before running; it can download models, install packages and start expensive training. See `docs/experiment_catalogue.md` and the new pipeline for supported experiments.


In [ ]:
from transformers import (
    GPT2Tokenizer, 
    GPT2Model, GPT2Config, 
    GPT2ForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
import torch
import torch.nn.functional as F
import pandas as pd
import torch.nn as nn
from sklearn.preprocessing import MultiLabelBinarizer
from datasets import Dataset, load_dataset
import pandas as pd
import os
import numpy as np
from peft import TaskType, LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from transformers import Trainer

import torch
print(torch.cuda.is_available())

In [ ]:
pip install transformers torch torchvision datasets peft scikit-learn

In [ ]:
# ID to article mapping
id_mapping = {
    0: 'Article 1: Obligation to respect human rights',
    1: 'Article 2: Right to life',
    2: 'Article 3: Prohibition of torture',
    3: 'Article 4: Prohibition of slavery and forced labour',
    4: 'Article 5: Right to liberty and security',
    5: 'Article 6: Right to a fair trial',
    6: 'Article 7: No punishment without law',
    7: 'Article 8: Right to respect for private and family life',
    8: 'Article 9: Freedom of thought, conscience and religion',
    9: 'Article 10: Right to freedom of speech and expression',
    10: 'No Articles Violated'
}

# Article to ID mapping
article_mapping = {v: k for k, v in id_mapping.items()}

# Load data from parquet files
train = pd.read_parquet('ecthr_a_article_desc_train.parquet')
test = pd.read_parquet('ecthr_a_article_desc_test.parquet')
validation = pd.read_parquet('ecthr_a_article_desc_validation.parquet')

# Convert Pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train)
test_dataset = Dataset.from_pandas(test)
validation_dataset = Dataset.from_pandas(validation)


In [ ]:
# Data processing: Rename columns
train_dataset = train_dataset.rename_column("labels", "original_labels")
test_dataset = test_dataset.rename_column("labels", "original_labels")
validation_dataset = validation_dataset.rename_column("labels", "original_labels")

# Function to map articles to their corresponding IDs
def map_articles(articles):
    return [article_mapping[article] for article in articles]

# Apply map_articles function to the 'articles' column
def apply_mapping(examples):
    examples['mapped_labels'] = map_articles(examples['articles'])
    return examples

# Apply mapping to each dataset
train_dataset = train_dataset.map(apply_mapping)
test_dataset = test_dataset.map(apply_mapping)
validation_dataset = validation_dataset.map(apply_mapping)

In [ ]:
# Initialize the MultiLabelBinarizer with the classes (range from 0 to 10 because there are 11 labels)
mlb = MultiLabelBinarizer(classes=range(11))

# Function to apply one-hot encoding using MultiLabelBinarizer
def one_hot_encode_labels(examples):
    one_hot_labels = mlb.fit_transform(examples['mapped_labels'])
    examples['one_hot_labels'] = [label.astype(int) for label in one_hot_labels]
    return examples

# Apply one-hot encoding to each dataset
train_dataset = train_dataset.map(one_hot_encode_labels, batched=True)
test_dataset = test_dataset.map(one_hot_encode_labels, batched=True)
validation_dataset = validation_dataset.map(one_hot_encode_labels, batched=True)

In [ ]:
# Remove unwanted columns
train_dataset = train_dataset.remove_columns(['original_labels', 'articles', 'mapped_labels'])
test_dataset = test_dataset.remove_columns(['original_labels', 'articles', 'mapped_labels'])
validation_dataset = validation_dataset.remove_columns(['original_labels', 'articles', 'mapped_labels'])

# Rename the column to match what the model expects
train_dataset = train_dataset.rename_column("one_hot_labels", "labels")
test_dataset = test_dataset.rename_column("one_hot_labels", "labels")
validation_dataset = validation_dataset.rename_column("one_hot_labels", "labels")

print("Train dataset columns:", train_dataset.column_names)
print("Test dataset columns:", test_dataset.column_names)
print("Validation dataset columns:", validation_dataset.column_names)

In [ ]:
# Tokenizer initialization
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialogRPT-updown")

# Processing function to tokenize input text
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)

# Apply tokenization to each dataset
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)
validation_dataset = validation_dataset.map(tokenize_function, batched=True)

In [ ]:
# Remove the text column to avoid feeding raw text to the model
train_dataset = train_dataset.remove_columns(['text'])
test_dataset = test_dataset.remove_columns(['text'])
validation_dataset = validation_dataset.remove_columns(['text'])

# Ensure datasets are set up for PyTorch (if using PyTorch)
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
validation_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [ ]:
print("Processed train dataset:", train_dataset)
print("Processed test dataset:", test_dataset)
print("Processed validation dataset:", validation_dataset)

In [ ]:
print("Any NaN in train_dataset['input_ids']? ",np.isnan(np.array(train_dataset['input_ids'])).any())
print("Any NaN in train_dataset['attention_mask']? ",np.isnan(np.array(train_dataset['attention_mask'])).any())
print("-----"*10)
print("Any NaN in test_dataset['input_ids']? ",np.isnan(np.array(test_dataset['input_ids'])).any())
print("Any NaN in test_dataset['attention_mask']? ",np.isnan(np.array(test_dataset['attention_mask'])).any())


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    # Apply sigmoid to get probabilities
    predictions = sigmoid(logits)
    
    # Convert probabilities to binary predictions (multi-label)
    predictions = (predictions > 0.5).astype(int)
    
    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    f1_micro = f1_score(labels, predictions, average='micro', zero_division=0)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)
    f1_weighted = f1_score(labels, predictions, average='weighted', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }


In [ ]:
# Initialize the model for sequence classification with multi-label problem type
num_labels = 11
model = GPT2ForSequenceClassification.from_pretrained(
    "microsoft/DialogRPT-updown",
    num_labels=num_labels,
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True  # Add this argument
)

In [ ]:
# Custom Trainer class to use BCEWithLogitsLoss for multi-label classification
class MultiLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        labels = labels.to(model.device)
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.BCEWithLogitsLoss()
        loss = loss_fct(logits.view(-1, self.model.config.num_labels),
                        labels.float().view(-1, self.model.config.num_labels))
        return (loss, outputs) if return_outputs else loss


In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    overwrite_output_dir= True,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate= 3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    #max_steps = 50,
    max_grad_norm=1.0,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none",
    seed=42,
    metric_for_best_model = "f1_micro",
    greater_is_better = True
)

In [ ]:
import torch
from transformers import get_linear_schedule_with_warmup

def create_optimizer_and_scheduler(model, args, num_training_steps):
    no_decay = ['bias', 'LayerNorm.weight']
    
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            'weight_decay': args.weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0
        }
    ]
    
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=args.learning_rate, eps=args.adam_epsilon)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=args.warmup_steps, num_training_steps=num_training_steps)
    
    return optimizer, scheduler

num_training_steps = (len(train_dataset) // training_args.per_device_train_batch_size) * training_args.num_train_epochs
optimizer, scheduler = create_optimizer_and_scheduler(model, training_args, num_training_steps)

In [ ]:
trainer = MultiLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    optimizers=(optimizer,scheduler)
)

In [ ]:
trainer.train()